In [ ]:
ALLOW_DOWNLOADS = False

def download_with_permission(operation, *args, **kwargs):
    if not ALLOW_DOWNLOADS:
        raise RuntimeError("Для загрузки ресурсов установите ALLOW_DOWNLOADS = True; пути к данным описаны в INPUTS.md.")
    return operation(*args, **kwargs)


# Практическое задание

# Работа с эмбеддингами

## курс "Анализ неструктурированных данных"


## Введение

В этом задании вы научитесь использовать векторные представления слов для выделения слов, схожих по смыслу

### Обязательные библиотеки:
- [Gensim](https://radimrehurek.com/gensim/) — инструмент для решения различных задач NLP (тематическое моделирование, представление текстов, ...);
- [Numpy](http://www.numpy.org) — библиотека для векторных вычислений;
Для выполнения дополнительных заданий (по желанию)
- [Pandas](https://pandas.pydata.org) - библиотека для анализа табличных данных;
- [scikit-learn](http://scikit-learn.org/stable/index.html) — библилиотека с алгоритмами классического машинного обучения;
- [matplotlib](https://matplotlib.org) - библиотека для построения графиков;

-------

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

# Константы
DATA_DIR = 'data'
RANDOM_STATE = 42

### Модель, генерирующая вектора слов

Для решения вам потребуются предобученная модель векторных представлений слов.

Будем использовать русскоязычную [модель эмбеддингов](https://rusvectores.org/ru/models/) (Если функция ниже не работает - [прямая ссылка на скачивание](http://vectors.nlpl.eu/repository/20/213.zip)), обученную на GeoWAC - мультиязычном корпусе интернет-страниц.
Скачаем ее (внимание! должен быть загружен модуль utils.py - как загружать см. readme)

In [ ]:
from utils import download_model

download_with_permission(download_model, data_dir=DATA_DIR)

## Часть 1. Предобученные векторные представления слов

Скачайте предобученные вектора и загрузите их с помощью функции [fasttext.FastTextKeyedVectors.load](https://radimrehurek.com/gensim/models/fasttext.html) библиотеки Gensim.

In [ ]:
import gensim

wv_embeddings = gensim.models.fasttext.FastTextKeyedVectors.load(
    DATA_DIR + "/model/model.model"
)

В результате в переменной `wv_embeddings` лежит FastText модель.
Это улучшенная версия Word2Vec моделей, отличающаяся в том, что модель работает с подсловами (character n-grams), а не целыми словами. Это дает несколько важных преимуществ:

1. Модель способна работать с неизвестными словами, которых не было в обучающей выборке
2. Лучше справляется со словами с опечатками
3. Эффективна для языков с богатой морфологией (включая русский)

В результате с помощью нее можно генерировать эмбеддинги произвольных слов, которые иначе были бы out of vocabulary.

Посмотрим, как "выглядит" модель: распечатаем эмбеддинг чего угодно.
На вход модели можно подавать как известные слова, так и любые неизвестные (любые сочетания символов), эмбеддинг будет построен.

In [ ]:
wv_embeddings['студент']

Учтите, что эти эмбеддинги не нормированы (так как получаются суммированием биграмм) и для корректной работы функций их желательно нормировать. Это можно сделать вот так:

In [ ]:
emb = np.array(wv_embeddings['студент'])
emb /= np.linalg.norm(emb)
emb

### Методы для работы с этими векторами, встроенные в библиотеку Gensim
Все методы используют векторную арифметику: получают вектор и выбирают слова с близкими векторами. Термин "семантическая близость" здесь следует понимать не иначе как "близкое расстояние между векторами"

### **most_similar_to_given:**

Метод находит из заданного списка слово, наиболее  близкое к целевому слову.

Принимает два параметра:
- word - Целевое слово для сравнения (str)
- list_words - Список слов-кандидатов для сравнения (list of str)

Возвращает строку - слово из списка с наибольшей косинусной близостью к целевому слову.

Пример: найти с помощью данного метода наиболее похожее на слово музыка слово из списка ['вода', 'звук', 'рюкзак', 'мышь'].



In [ ]:
wv_embeddings.most_similar_to_given('музыка', ['вода', 'звук', 'рюкзак', 'мышь'] )

### **doesnt_match:**

Метод находит "белую ворону" - слово, которое  наименее похоже на остальные в представленном списке.

Принимает единственный параметр:

    words - Список слов для сравнения (list of str)

Возвращает строку - слово, которое наименее похоже на остальные слова из списка.

Пример: найти с помощью данного метода слово, которое является лишним в списке:'завтрак', 'обед', 'ужин', 'студент'

In [ ]:
wv_embeddings.doesnt_match(['завтрак', 'обед', 'ужин', 'студент'])

## **most_similar:**

Метод, находящий слова, наиболее близкие к заданным положительным примерам и далекие от отрицательных.

Принимает 2 параметра-списка (списки могут быть разной длины, непустые):

    positive - Список из строк, векторы которых будут считаться "позитивами" при поиске
    negative - Список из строк, векторы которых будут считаться "негативами" при поиске

Возвращаее список из пар (слово, косинусная близость), отсортированных по убыванию близости

Пример: найти с помощью данного метода слово, похожее на слова из названия нашего факультета и непохожее на названия других наук.

In [ ]:
wv_embeddings.most_similar([ 'вычисления', 'математика','кибернетика', 'информатика'],['физика','химия', 'биология', ])